In [1]:
import json
from pathlib import Path
import pandas as pd

# =========================
# 1) 配置：改成你的 4 个 json 路径
# =========================
json_paths = [
    "initial_sampling_Co_Mo.json",
    "initial_sampling_Co_V.json",
    "initial_sampling_Co_Ni.json",
    "initial_sampling_Fe_Ni.json",
]

out_xlsx = "BO_data.xlsx"

# =========================
# 2) 工具函数
# =========================
def load_json(p: str):
    p = Path(p)
    with p.open("r", encoding="utf-8") as f:
        return json.load(f)

def to_dataframe(obj, source_name: str):
    """
    将 JSON 转成 DataFrame，并尽可能“拉平”嵌套结构。
    - 若 JSON 顶层是 list[dict]：直接 json_normalize
    - 若 JSON 顶层是 dict：
        - 若含有某个 list 值，优先对该 list 做 json_normalize
        - 否则把 dict 当一行记录
    """
    # list -> 表
    if isinstance(obj, list):
        df = pd.json_normalize(obj, sep=".")
        df.insert(0, "source", source_name)
        return df

    # dict -> 尝试找一个“像记录列表”的字段
    if isinstance(obj, dict):
        list_keys = [k for k, v in obj.items() if isinstance(v, list)]
        if list_keys:
            # 选“最长的 list”作为主记录表
            main_key = max(list_keys, key=lambda k: len(obj[k]))
            df = pd.json_normalize(obj[main_key], sep=".")
            # 把 dict 里其他非 list 的字段当作元信息追加到每一行
            meta = {k: v for k, v in obj.items() if k != main_key and not isinstance(v, (list, dict))}
            for k, v in meta.items():
                df[k] = v
            df.insert(0, "source", source_name)
            df.insert(1, "json_main_key", main_key)
            return df
        else:
            # 没有 list：当作单条记录
            df = pd.json_normalize(obj, sep=".")
            df.insert(0, "source", source_name)
            return df

    # 其他类型：兜底
    return pd.DataFrame([{"source": source_name, "value": str(obj)}])

def safe_sheet_name(name: str) -> str:
    """Excel sheet 名不能超过 31 字符，且不能含某些字符。"""
    bad = ['\\', '/', '*', '?', ':', '[', ']']
    for ch in bad:
        name = name.replace(ch, "_")
    return name[:31] if len(name) > 31 else name

# =========================
# 3) 读取 -> DataFrame
# =========================
dfs = []
sheet_map = {}  # sheet_name -> df

for p in json_paths:
    p = Path(p)
    obj = load_json(str(p))
    df = to_dataframe(obj, source_name=p.stem)

    # 记录
    dfs.append(df)

    # 单独 sheet
    sheet_name = safe_sheet_name(p.stem)
    sheet_map[sheet_name] = df

# 合并总表（列不一致会自动 union）
merged = pd.concat(dfs, ignore_index=True, sort=False)

# =========================
# 4) 写入 Excel
# =========================
out_xlsx = Path(out_xlsx)
out_xlsx.parent.mkdir(parents=True, exist_ok=True)

with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
    # 总表
    merged.to_excel(writer, index=False, sheet_name="all_merged")

    # 每个 JSON 一个 sheet
    for sheet_name, df in sheet_map.items():
        df.to_excel(writer, index=False, sheet_name=sheet_name)

print(f"[OK] Saved to: {out_xlsx}")
print(f"[OK] all_merged rows: {len(merged)}  cols: {len(merged.columns)}")


[OK] Saved to: BO_data.xlsx
[OK] all_merged rows: 4  cols: 673


In [3]:
import json
from pathlib import Path
import pandas as pd

# =========================
# 1) 配置：改成你的 4 个 json 路径
# =========================
json_paths = [
    "initial_sampling_Co_Mo.json",
    "initial_sampling_Co_V.json",
    "initial_sampling_Co_Ni.json",
    "initial_sampling_Fe_Ni.json",
]

out_xlsx = "BO_data.xlsx"

# =========================
# 2) 读取并转成 DataFrame（适配你这种“columns-orient”结构）
# =========================
def read_columns_oriented_json(p: str) -> pd.DataFrame:
    p = Path(p)
    with p.open("r", encoding="utf-8") as f:
        obj = json.load(f)

    # 你的结构：{"col1": {"0": v0, "1": v1, ...}, "col2": {...}, ...}
    # 直接 DataFrame(obj) 就会得到 index=行号（字符串）+ columns=列名
    df = pd.DataFrame(obj)

    # index 变成真正的整数并排序（更像表）
    try:
        df.index = df.index.astype(int)
        df = df.sort_index()
    except Exception:
        pass

    df = df.reset_index(drop=False).rename(columns={"index": "row_id"})
    df.insert(0, "source_file", p.stem)  # 标记来源文件名
    return df

def safe_sheet_name(name: str) -> str:
    bad = ['\\', '/', '*', '?', ':', '[', ']']
    for ch in bad:
        name = name.replace(ch, "_")
    return name[:31]  # Excel sheet 名最大 31

dfs = []
sheet_map = {}

for jp in json_paths:
    df = read_columns_oriented_json(jp)
    dfs.append(df)
    sheet_map[safe_sheet_name(Path(jp).stem)] = df

merged = pd.concat(dfs, ignore_index=True, sort=False)

# =========================
# 3) 写入 Excel：每个 JSON 一个 sheet + 汇总 all_merged
# =========================
out_xlsx = Path(out_xlsx)
out_xlsx.parent.mkdir(parents=True, exist_ok=True)

with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
    merged.to_excel(writer, index=False, sheet_name="all_merged")
    for sheet_name, df in sheet_map.items():
        df.to_excel(writer, index=False, sheet_name=sheet_name)

print(f"[OK] Saved: {out_xlsx}")
print(f"[OK] all_merged rows={len(merged)}, cols={len(merged.columns)}")


[OK] Saved: BO_data.xlsx
[OK] all_merged rows=128, cols=23
